# 1. Extracción de metadatos y validación del OCR

Este cuaderno reproduce la construcción de la base de datos de la tesis (Sección 4.1 y Anexo A): el texto de los artículos obtenido por OCR, la extracción de sus metadatos, la validación contra la base de referencia y la base final de 183 artículos.

La descarga de los PDF y el OCR no se ejecutan por defecto, porque requieren los archivos de la revista, Tesseract 5 y Poppler. Sus resultados están en `data/interim/`, de modo que el resto del cuaderno corre sin ellos.

In [1]:
import sys, json
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))
import numpy as np
import pandas as pd
from IPython.display import display
from chjs import rutas
from chjs.cuadernos import ejecutar, figuras
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

## Descarga y OCR

Los PDF se descargan del sitio de la revista (`chjs.descarga`). Cada página se convierte en una imagen de 400 ppp y se lee con Tesseract en inglés, español y portugués (`chjs.ocr`), hasta la página que contiene las palabras clave. Una segunda pasada (`scripts/ocr_superindices.py`) vuelve a leer, ampliados, los superíndices de la línea de autores, que en la primera pasada se confundían con signos de puntuación.

Además de los 183 artículos, el texto OCR incluye las presentaciones de cada número y los editoriales, que no forman parte de la población de estudio.

In [2]:
EJECUTAR_OCR = False                     # requiere los PDF en data/raw/, Tesseract 5 y Poppler
if EJECUTAR_OCR:
    from chjs.descarga import descargar_pdfs
    from chjs.ocr import ocr_lote
    descargar_pdfs(rutas.CRUDOS)
    ocr_lote(sorted(rutas.CRUDOS.glob("*.pdf")), rutas.OCR_TEXTO)
    print(ejecutar("ocr_superindices.py"))

ocr = json.load(open(rutas.OCR_TEXTO, encoding="utf-8"))
sup = json.load(open(rutas.SUPERINDICES, encoding="utf-8"))
print(f"{len(ocr)} documentos con texto OCR; {len(sup)} con la segunda pasada de superíndices")

197 documentos con texto OCR; 189 con la segunda pasada de superíndices


## Extracción de metadatos

Las reglas de `chjs.extraccion` identifican en el texto el título, el resumen, las palabras clave, los autores y las afiliaciones. Las afiliaciones se numeran por orden de aparición y cada autor se vincula con la suya mediante los superíndices. Por ejemplo:

In [3]:
from chjs.extraccion import extraer
doc = next(o for o in ocr if "Alzaid" in o["archivo"])
meta = extraer(doc["texto"], sup.get(doc["archivo"], {}).get("grupos"))
print("Título:        ", meta["titulo"])
print("Palabras clave:", meta["palabras_clave"])
print("Afiliaciones:  ", meta["afiliaciones"])
pd.DataFrame(meta["autores"])

Título:         Bernoulli Difference Time Series Models
Palabras clave: ['bernoulli difference distribution', 'jacobs-lewis model', 'markov model', 'stock market', 'time series model']
Afiliaciones:   {'unica': 'Department of Statistics and Operations Research, King Saud University, Riyadh, Kingdom of Saudi Arabia'}


,nombre,indices,afiliaciones,paises
0,Abdulhamid A. Alzaid,[],"[Department of Statistics and Operations Research, King Saud University, Riyadh, Kingd...",[Saudi Arabia]
1,Maha A. Omair,[*],"[Department of Statistics and Operations Research, King Saud University, Riyadh, Kingd...",[Saudi Arabia]
2,Weaam. M. Alhadlaq,[],"[Department of Statistics and Operations Research, King Saud University, Riyadh, Kingd...",[Saudi Arabia]


## Validación de la extracción (Tabla 4.3)

Cada campo extraído se compara con la base de referencia de la primera etapa del estudio, construida con la capa de texto de los PDF y una revisión manual. Los artículos se emparejan uno a uno por título (asignación húngara) y los autores de cada artículo, por nombre. La tabla muestra el porcentaje de concordancia con la primera pasada del OCR y con las dos pasadas.

In [4]:
from chjs.validacion import comparar
CAMPOS = {"titulo": "Título", "resumen": "Resumen", "palabras_clave": "Palabras clave",
          "n_autores": "Número de autores", "autores": "Nombres de los autores",
          "afiliaciones_articulo": "Afiliaciones del artículo", "paises_articulo": "Países del artículo",
          "afiliacion_autor": "Afiliación de cada autor", "pais_autor": "País de cada autor"}
ref = json.load(open(rutas.BASE_REFERENCIA, encoding="utf-8"))
concordancia = {}
for nombre, s in [("Primera pasada", None), ("Dos pasadas", sup)]:
    df = comparar(ref, ocr, s)
    concordancia[nombre] = (100 * df[list(CAMPOS)].mean()).round(1)
pd.DataFrame(concordancia).rename(index=CAMPOS)

,Primera pasada,Dos pasadas
Título,99.5,99.5
Resumen,100.0,100.0
Palabras clave,99.8,99.8
Número de autores,100.0,100.0
Nombres de los autores,99.8,99.8
Afiliaciones del artículo,100.0,100.0
Países del artículo,99.5,99.5
Afiliación de cada autor,86.1,99.5
País de cada autor,90.5,99.1


Las discrepancias que quedan se listaron una a una (`scripts/listar_discrepancias.py`) y se revisaron contra la imagen de la primera página de cada artículo. La mayoría son diferencias de grafía en los nombres de los autores, como las tildes de los nombres escritos en versalitas.

In [5]:
ejecutar("listar_discrepancias.py")
disc = pd.read_csv(rutas.VALIDACION / "discrepancias.csv")
print(f"{len(disc)} discrepancias en {disc['archivo'].nunique()} artículos")
disc.groupby("campo").size().rename("discrepancias")

59 discrepancias en 46 artículos


campo
afiliacion_autor     3
autor                2
autor_grafia        51
palabra_clave        2
titulo               1
Name: discrepancias, dtype: int64

## Base final

Las correcciones confirmadas en la revisión (`chjs.correcciones`) se aplican sobre la base de referencia, junto con la normalización de instituciones (`chjs.instituciones`) y de nombres de autores (`chjs.autores`). El resultado es idéntico a `data/processed/base_final.json`.

In [6]:
from chjs.correcciones import construir, resumen
base, cambios = construir(ref)
final = json.load(open(rutas.BASE_FINAL, encoding="utf-8"))
print("Idéntica a data/processed/base_final.json:", base == final)
print(resumen(base))
cambios.groupby("campo").size().rename("cambios")

Idéntica a data/processed/base_final.json: True
{'autores': 396, 'afiliaciones_originales': 340, 'instituciones': 204, 'paises': 44}


campo
afiliacion_agregada      1
nombre                 121
pais                     4
universidad            192
Name: cambios, dtype: int64

La normalización reduce 340 formas de escritura de las afiliaciones a 204 instituciones. La Universidade de São Paulo, por ejemplo, aparece escrita de las siguientes formas:

In [7]:
usp = sorted({a["universidad_raw"] for r in base for a in r["Autores"]
              if a.get("universidad") == "Universidade de São Paulo"})
print(len(usp), "formas distintas")
usp[:10]

20 formas distintas


['Departamento de Estatística, Universidade de São Paulo, Brazil',
 'Departamento de Estatística, Universidade de São Paulo, São Paulo, Brazil',
 'Departamento de Matemática Aplicada e Estatística, ICMC, USP - Universidade de São Paulo, São Carlos, SP, Brasil',
 'Departamento de Matemática Aplicada e Estatística, Universidade de São Paulo, Brasil',
 'Department of App. Math. and Stat., Universidade de São Paulo, São Carlos-SP, Brazil',
 'Department of Applied Mathematics & Statistics, ICMC, University of São Paulo, São Carlos, SP, Brazil',
 'Department of Applied Mathematics and Statistics, University of São Paulo, SP, Brazil',
 'Department of Applied Mathematics and Statistics, University of São Paulo, São Carlos, Brazil',
 'Department of Exact Sciences, University of São Paulo, Piracicaba, Brazil',
 'Department of Exact Sciences, University of São Paulo, São Paulo, Brazil']

## Artículos por año (Tabla 4.1)

In [8]:
import re
anios = pd.DataFrame([{"Año": int(re.search(r"\d{4}", r["fecha"]).group()), "Volumen": r["Volumen"],
                       "Número": r["numero"]} for r in base])
t41 = anios.groupby("Año").agg(Volumen=("Volumen", "first"), Números=("Número", "nunique"),
                               Artículos=("Número", "size"))
print("Total:", t41["Artículos"].sum(), "artículos en", t41["Números"].sum(), "números")
t41

Total: 183 artículos en 32 números


,Volumen,Números,Artículos
Año,,,
2010,1,2,13
2011,2,2,14
2012,3,2,14
2013,4,2,14
2014,5,2,13
2015,6,2,13
2016,7,2,10
2017,8,2,9
2018,9,2,9
